### =============================================================================
### HOUSE PRICE PREDICTION - MODEL TRAINING & EVALUATION
### =============================================================================
### Author: Beksultan a.k.a rsuvbe
### Date: 23.03.2026
### Description: Preprocessing, training Linear Regression model, and evaluation
### =============================================================================



In [ ]:
# # 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, cross_validate
import joblib
import os 
import json

# 1. Plot Settings 
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 2. Load Data

# Loading training data

train_df = pd.read_csv('../data/train.csv')

print(f"Data loaded: {train_df.shape[0]} rows x {train_df.shape[1]} columns")

X = train_df.drop(['SalePrice', 'Id'], axis = 1)
y = train_df['SalePrice']

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")

pd.set_option("display.max_rows", None)

train_df = train_df.drop(columns=["PoolQC", "MiscFeature", "Fence", "Alley"])


inf_df = pd.DataFrame({
    "NotNull": train_df.notnull().sum(),
    "Dtype": train_df.dtypes
})


In [ ]:
## 3. Target Variable Transformation

# Log transformation (due to high skewness: 1.88)

print("\n" + "="*60)
print("TARGET VARIABLE TRANSFORMATION")
print("="*60)

y_log = np.log1p(y)

print(f"Original SalePrice - Mean: ${y.mean():,.0f}, Skew: {y.skew():.2f}")
print(f"Log-transformed    - Mean: {y_log.mean():.2f}, Skew: {y_log.skew():.2f}")

# Visualize 
fig, axes = plt.subplots(1,2, figsize=(14,5))

axes[0].hist(y, bins=50, edgecolor = 'black', alpha=0.7)
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Original SalePrice Distribution')
axes[0].grid(True,alpha=0.3)

axes[1].hist(y_log, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Log(SalePrice)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed SalePrice Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/06_target_transformation.png', dpi = 150)
plt.show()

In [ ]:
# # 4. Identify Column Types

# Identify numeric and categorical columns

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"\nNumeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")


In [ ]:
train_df["TotalSF"] = train_df["1stFlrSF"] + train_df["2ndFlrSF"] + train_df["TotalBsmtSF"]

train_df["HouseAge"] = train_df["YrSold"] - train_df["YearBuilt"]

train_df["RemodAge"] = train_df["YearRemodAdd"] - train_df["YearBuilt"]

train_df["TotalBath"] = train_df["BsmtFullBath"] + train_df["BsmtHalfBath"] + train_df["FullBath"] + train_df["HalfBath"]

In [ ]:
for col in ["TotalSF", "HouseAge", "RemodAge", "TotalBath"]:
    print(col, train_df[col].corr(train_df["SalePrice"]))

In [ ]:
# # 5. Create Preprocessing Pipeline

# Pipeline for Numeric features

numeric_transformer = Pipeline(steps=[
	('imputer', SimpleImputer(strategy='median')), 
	('scaler', StandardScaler()) 
])

# Pipeline for Categorical features

categorical_transformer = Pipeline(steps=[
	('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
	('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

#Combine preprocessing steps 
preprocessor = ColumnTransformer(transformers=[
	('num', numeric_transformer, numeric_cols),
	('cat', categorical_transformer, categorical_cols)
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

param_grids = {
    'Ridge': {'regressor__alpha': [0.001, 0.01, 0.1, 1, 10]},
    'Lasso': {'regressor__alpha': [0.0001, 0.001, 0.01, 0.1]},
    'LinearRegression': {}  
}

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso()
}

results={}

print("Preprocessing pipeline created")

In [ ]:
from sklearn.model_selection import GridSearchCV

# 6. Split data into Train/Validation Sets
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42, shuffle=True
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Validation set: {X_test.shape[0]} samples")

# 7. Model Training and Evaluation via Grid Search
for name, model in models.items():
    # Combine preprocessing steps and the model into a single pipeline
    current_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # Setup GridSearchCV for hyperparameter tuning with cross-validation
    grid_search = GridSearchCV(
        current_pipeline,
        param_grids[name],
        cv=kf,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    # Train the pipeline on the training data
    grid_search.fit(X_train, y_train_log)

    # Make predictions on the test set (still in log scale)
    y_pred_log = grid_search.predict(X_test)

    # Convert predictions and target values back to the original scale (dollars)
    y_pred_orig = np.expm1(y_pred_log)
    y_test_orig = np.expm1(y_test_log)

    # Evaluate model performance using RMSE in dollars and R^2 score
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
    r2 = r2_score(y_test_orig, y_pred_orig)

    # Save metrics and best parameters for comparison
    results[name] = {
        'best_params': grid_search.best_params_,
        'rmse_dollars': rmse,
        'r2_score': r2,
        'best_estimator': grid_search.best_estimator_
    }

    print(f"RMSE in dollars with {name}: {rmse:.2f}")
    print(f"R^2 with {name}: {r2:.4f}")
    print(f"Best parameters for {name}: {grid_search.best_params_}")


# 8. Extract feature names after preprocessing
numeric_features_names = numeric_cols

# Retrieve the fitted preprocessor instance from the best estimator
trained_preprocessor = grid_search.best_estimator_.named_steps['preprocessor']

# Extract expanded feature names from the One-Hot Encoder
ohe = trained_preprocessor.named_transformers_['cat'].named_steps['onehot']
categorical_feature_names = ohe.get_feature_names_out(categorical_cols).tolist()

# Combine numeric and one-hot encoded categorical feature names
all_feature_names = numeric_features_names + categorical_feature_names

print(f"Total feature after preprocessing: {len(all_feature_names)}")

In [ ]:
# 9. Evaluate Model Performance & Check for Overfitting
# (Can be run for the best model from grid_search)

best_model = grid_search.best_estimator_

# Make predictions on train and test sets
y_train_pred_log = best_model.predict(X_train)
y_test_pred_log = best_model.predict(X_test)

# Convert back from log scale to the original scale (dollars)
y_train_pred = np.expm1(y_train_pred_log)
y_test_pred = np.expm1(y_test_pred_log)

y_train_orig = np.expm1(y_train_log)
y_test_orig = np.expm1(y_test_log)

# Function to calculate metrics
def calculate_metrics(y_true, y_pred, label):
    """Calculate and print regression metrics"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{label} SET:")
    print(f" RMSE: ${rmse:,.0f}")
    print(f" MAE: ${mae:,.0f}")
    print(f" R^2: {r2:.4f}")

    return {'rmse': rmse, 'mae': mae, 'r2': r2}

train_metrics = calculate_metrics(y_train_orig, y_train_pred, "TRAIN")
test_metrics = calculate_metrics(y_test_orig, y_test_pred, "TEST")

# Check for overfitting
print("="*60)
print("OVERFITTING CHECK")
print("="*60)

r2_diff = train_metrics['r2'] - test_metrics['r2']
if r2_diff > 0.1:
    print(f"WARNING: Possible Overfitting Detected!")
    print(f" R^2 Difference: {r2_diff:.4f}")
else:
    print(f"Model generalizes well")
    print(f" R^2 difference: {r2_diff:.4f}")

In [ ]:
# 10. Visualize Results for the Best Model

# Use y_test_orig and y_test_pred calculated in the previous evaluation block
y_val_original = y_test_orig
y_val_pred = y_test_pred

# Plot 1: Predictions vs Actual 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_val_original, y_val_pred, alpha=0.5, edgecolors='k', s=30)
axes[0].plot([y_val_original.min(), y_val_original.max()],
             [y_val_original.min(), y_val_original.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Sale Price ($)')
axes[0].set_ylabel('Predicted Sale Price ($)')
axes[0].set_title('Predictions vs Actual Values')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = y_val_original - y_val_pred

axes[1].scatter(y_val_pred, residuals, alpha=0.5, edgecolor='k', s=30)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Sale Price ($)')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/07_model_predictions.png', dpi=150)
plt.show()

# Plot 3: Residual Distribution 
plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Residual ($)')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/08_residual_distribution.png', dpi=150)
plt.show()

In [ ]:
# 11. Feature Importance Analysis (for Linear Models like LinearRegression, Ridge, Lasso)

# Get the best regressor from the pipeline's best estimator
best_regressor = grid_search.best_estimator_.named_steps['regressor']

# Check if the model uses coefficients (linear models)
if hasattr(best_regressor, 'coef_'):
    coefficients = best_regressor.coef_

    # Create DataFrame for feature importance
    feature_importance = pd.DataFrame({
        'feature': all_feature_names,
        'coefficients': coefficients,
        'abs_coefficients': np.abs(coefficients)
    }).sort_values('abs_coefficients', ascending=False)

    # Display top 20 most important features
    print("\n" + "="*60)
    print("TOP MOST IMPORTANT FEATURES")
    print("="*60)
    print(feature_importance.head(20).to_string(index=False))

    # Visualize top 15 features 
    plt.figure(figsize=(12, 10))
    top_15 = feature_importance.head(15)
    plt.barh(top_15['feature'], top_15['coefficients'], color='skyblue', edgecolor='navy')
    plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
    plt.xlabel('Coefficient Value')
    plt.title('Top 15 Feature Importance (Linear Model)')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('../results/09_feature_importance.png', dpi=150)
    plt.show()
else:
    print("The best model is tree-based and does not support coefficients (`coef_`).")

In [ ]:
# 12. SAVE MODEL AND ARTIFACTS

print("\n" + "="*60)
print("SAVING MODEL AND ARTIFACTS")
print("="*60)

os.makedirs('../models', exist_ok=True)

# 12.1 Select Best Model Based on Evaluation Results
best_model_name = max(results, key=lambda x: results[x]['r2_score'])
best_pipeline = results[best_model_name]['best_estimator']

print(f"\nBest model by R²: {best_model_name} (R² = {results[best_model_name]['r2_score']:.4f})")

# 12.2 Save the Best Pipeline and Preprocessor
model_filename = f'../models/{best_model_name.lower().replace(" ", "_")}_pipeline.pkl'
joblib.dump(best_pipeline, model_filename)

trained_preprocessor = best_pipeline.named_steps['preprocessor']
joblib.dump(trained_preprocessor, '../models/preprocessor.pkl')

print(f" Model Pipeline saved: {model_filename}")
print(f" Preprocessor saved: ../models/preprocessor.pkl")

# 12.3 Save Final Predictions
y_test_pred_log = best_pipeline.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)
y_test_orig = np.expm1(y_test_log)

final_rmse = np.sqrt(mean_squared_error(y_test_orig, y_test_pred))
final_mae = mean_absolute_error(y_test_orig, y_test_pred)
final_r2 = r2_score(y_test_orig, y_test_pred)

print(f"\n Final Test Metrics:")
print(f"  RMSE: ${final_rmse:,.0f}")
print(f"  MAE:  ${final_mae:,.0f}")
print(f"  R²:   {final_r2:.4f}")

# 12.4 Save Comprehensive Metrics
metrics = {
    'best_model': best_model_name,
    'selection_method': 'GridSearchCV with Cross-Validation',
    'best_params': results[best_model_name]['best_params'],
    'test_set': {
        'rmse': float(final_rmse),
        'mae': float(final_mae),
        'r2': float(final_r2),
        'n_samples': len(X_test)
    },
    'dataset': {
        'n_train_samples': int(X_train.shape[0]),
        'n_test_samples': int(X_test.shape[0]),
        'n_features_original': int(X.shape[1])
    }
}

with open('../results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f" Metrics saved: results/metrics.json")

# 12.5 Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

print(f"""
 Best Model: {best_model_name}
 Best Params: {results[best_model_name]['best_params']}
 Test R²: {final_r2:.4f}
 Test RMSE: ${final_rmse:,.0f}
 Model saved: {model_filename}
 Metrics saved: results/metrics.json

 PROJECT COMPLETE
""")